# Thí nghiệm 4-B — chạy lại PhoGPT bằng **đúng prompt của phiếu**

Lượt A (17/08) dùng prompt của TN3 và bọc qua `apply_chat_template`. Kết quả: mô hình
sao chép danh sách mã trong ví dụ, Hit@1 = 0,55%.

Nhưng phiếu TN4 mục 4.1 in một prompt **khác**, và mục 4.2 cảnh báo thẳng:
*"Nếu KHÔNG dùng đúng format, model có thể sinh output kém chất lượng."*
Vì lượt A không chạy đúng cấu hình phiếu chỉ định, chưa loại trừ được khả năng chính
việc thay prompt gây ra hiện tượng đó.

Notebook này chạy **đúng như phiếu viết**, chỉ đổi đúng hai thứ so với lượt A:

| | Lượt A (đã chạy) | Lượt B (notebook này) |
|---|---|---|
| Prompt | của TN3, không thẻ | **của phiếu §4.1**, có `### Câu hỏi:` / `### Trả lời:` |
| Cách bọc | `apply_chat_template` | **`tokenizer(prompt)` thô**, đúng §4.3 |
| Bộ chấm, dữ liệu, model, `max_new_tokens`, greedy | — | **giữ y nguyên** |

Giữ mọi thứ khác cố định để chỉ còn một biến. Cuối notebook có phần so sánh A ↔ B.

**Ba kết cục và ý nghĩa:**

| Kết cục | Ý nghĩa | Việc phải làm |
|---|---|---|
| B cũng sao chép ví dụ | Kết luận lượt A vững, không phải lỗi thay prompt | Giữ nguyên 0,55% trong Bảng 12 |
| B khác hẳn, Hit@1 cao hơn rõ | Lượt A sai vì thay prompt | **Thay số Bảng 12 bằng số của B** |
| B hỏng theo kiểu khác | Cần đọc kỹ output rồi báo thầy | Dừng, báo |

**Runtime:** T4 GPU. Cài ~3 phút, tải model ~5–8 phút, chạy 181 ca ~110 phút.

## 1. Kiểm GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), \
    "KHONG CO GPU. Vao Runtime -> Change runtime type -> T4 GPU roi chay lai."
p = torch.cuda.get_device_properties(0)
print(f"\nGPU : {p.name}")
print(f"VRAM: {p.total_memory/1e9:.2f} GB")

## 2. Cài thư viện — ghim `transformers==4.44.2`, xong phải **Restart session**

Giống hệt lượt A. Mã MPT của PhoGPT viết cho `transformers` đời 2023; bản mới làm gãy ở
ba chỗ. Cell dưới gỡ sạch trước khi cài để tránh trạng thái lẫn hai phiên bản.

In [ ]:
%pip uninstall -y -q transformers tokenizers
%pip install -q "transformers==4.44.2" accelerate bitsandbytes einops python-docx

import transformers, torch
print("transformers:", transformers.__version__, "(can 4.44.2)")
print("\n" + "=" * 68)
print("BAY GIO: Runtime -> Restart session, roi chay tiep tu Phan 2b.")
print("=" * 68)

### 2b. Chốt phiên bản — chạy **sau khi đã Restart**

In [ ]:
import transformers
V = transformers.__version__
print("transformers:", V)
if V != '4.44.2':
    raise SystemExit(f"transformers dang la {V}, can 4.44.2. Restart session roi chay lai cell nay.")
try:
    from transformers.utils import is_flax_available
    from transformers.models.llama.modeling_llama import LlamaDynamicNTKScalingRotaryEmbedding
    print("Moi truong sach. Chay tiep Phan 3.")
except ImportError as e:
    raise SystemExit(f"Cai dat con lan hai phien ban ({e}). Chay lai cell Phan 2 roi Restart.")

## 3. Nạp dữ liệu

Upload **`tn4b_input.zip`** — 10 tệp của gói TN4 cộng thêm `tn4a_perquery_phogpt.csv`
(kết quả lượt A) để Phần 12 so sánh A ↔ B.

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs('/content/tn4b_data', exist_ok=True)
up = files.upload()
for ten in up:
    if ten.endswith('.zip'):
        with zipfile.ZipFile(ten) as z:
            z.extractall('/content/tn4b_data')
    else:
        os.replace(ten, f'/content/tn4b_data/{ten}')
print()
for f in sorted(os.listdir('/content/tn4b_data')):
    print(' ', f)

## 4. Đọc và kiểm dữ liệu — 181 / 13.081 / 118–63

In [ ]:
MODEL_ID = 'vinai/PhoGPT-4B-Chat'

import os
import pandas as pd

DATA = '/content/tn4b_data'
OUT  = '/content/tn4b_ket_qua'
os.makedirs(OUT, exist_ok=True)

df_test = pd.read_csv(f'{DATA}/independent_scored_perquery.csv')
df_icd  = pd.read_csv(f'{DATA}/ICD10_cleaned.csv')

valid_codes_full  = set(df_icd['Mã ICD'].astype(str).str.strip().str.upper())
valid_codes_3char = set(c[:3] for c in valid_codes_full)

n_reach = int((df_test['voi_toi_duoc'] == True).sum())
n_outr  = int((df_test['voi_toi_duoc'] == False).sum())
print(f"So ca test          : {len(df_test):>6}   (ky vong 181)")
print(f"So ma ICD-10 day du : {len(valid_codes_full):>6}   (ky vong 13081)")
print(f"Reachable / out     : {n_reach} / {n_outr}   (ky vong 118 / 63)")
assert len(df_test) == 181,            "SAI so ca test"
assert len(valid_codes_full) == 13081, "SAI so ma ICD"
assert (n_reach, n_outr) == (118, 63), "SAI ty le reachable"

CO_LUOT_A = os.path.exists(f'{DATA}/tn4a_perquery_phogpt.csv')
print("\nCo ket qua luot A de so sanh:", "CO" if CO_LUOT_A else "KHONG (Phan 12 se rut gon)")

## 5. Prompt — **nguyên văn phiếu mục 4.1**

Khác prompt lượt A đúng bốn chỗ, tất cả đều theo phiếu:

1. mở đầu bằng thẻ `### Câu hỏi:`
2. kết thúc bằng thẻ `### Trả lời:`
3. `Đọc mô tả triệu chứng bệnh nhân **dưới đây**, đưa ra…` (lượt A: `NHIỆM VỤ: Đọc mô tả…`)
4. quy tắc 4 là `Chỉ trả về JSON.` (lượt A: `Chỉ trả về JSON **theo mẫu**.`)

Cell dưới tự đối chiếu: gom mọi khoảng trắng rồi so với bản in trong phiếu, lệch là dừng.

In [ ]:
PROMPT_PHIEU = """### Câu hỏi:

Bạn là bác sĩ trợ lý chuyên chẩn đoán bằng mã ICD-10.

Đọc mô tả triệu chứng bệnh nhân dưới đây, đưa ra danh sách TOP-10 mã ICD-10
có khả năng nhất, sắp xếp theo xác suất giảm dần.

QUY TẮC:
1. Mỗi mã ICD-10 phải là mã HỢP LỆ (định dạng: 1 chữ cái + 2-3 chữ số +
   tùy chọn "." + 1-2 chữ số. Ví dụ: A00, B07.9, K21.9).
2. Được phép dùng BẤT KỲ mã nào trong toàn bộ catalogue ICD-10 của
   Bộ Y tế Việt Nam (Quyết định 4469/QĐ-BYT 2020), gồm khoảng 13.081 mã.
3. Đưa mã 3 ký tự (phân nhóm) nếu không đủ tự tin về ký tự thứ 4.
4. KHÔNG giải thích. Chỉ trả về JSON.

VÍ DỤ:
Mô tả: "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ, đã 3 ngày."
Kết quả JSON: {{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Mô tả: "{query_text}"
Kết quả JSON:

### Trả lời:"""

import re, hashlib
_gom = lambda s: re.sub(r'\s+', ' ', s).strip()

# Ban in trong phieu muc 4.1 (trich tu docx; khoang trang da bi gom khi trich xuat)
_PHIEU_GOC = ('### Câu hỏi:  Bạn là bác sĩ trợ lý chuyên chẩn đoán bằng mã ICD-10.  Đọc mô tả '
 'triệu chứng bệnh nhân dưới đây, đưa ra danh sách TOP-10 mã ICD-10  có khả năng nhất, sắp '
 'xếp theo xác suất giảm dần.  QUY TẮC: 1. Mỗi mã ICD-10 phải là mã HỢP LỆ (định dạng: 1 chữ '
 'cái + 2-3 chữ số +     tùy chọn "." + 1-2 chữ số. Ví dụ: A00, B07.9, K21.9). 2. Được phép '
 'dùng BẤT KỲ mã nào trong toàn bộ catalogue ICD-10 của     Bộ Y tế Việt Nam (Quyết định '
 '4469/QĐ-BYT 2020), gồm khoảng 13.081 mã. 3. Đưa mã 3 ký tự (phân nhóm) nếu không đủ tự tin '
 'về ký tự thứ 4. 4. KHÔNG giải thích. Chỉ trả về JSON.  VÍ DỤ: Mô tả: "Em bị đau bụng nhiều, '
 'buồn nôn, sốt nhẹ, đã 3 ngày." Kết quả JSON: {{"top10_icd": ["K35.9", "K52.9", "K37", '
 '"K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}  --- Mô tả: "{query_text}" '
 'Kết quả JSON:  ### Trả lời:')

assert _gom(PROMPT_PHIEU) == _gom(_PHIEU_GOC), \
    "Prompt KHONG khop ban in trong phieu - dung lai, kiem tay."
print("Prompt khop nguyen van phieu muc 4.1 (sau khi gom khoang trang).")
print("SHA-256:", hashlib.sha256(PROMPT_PHIEU.encode()).hexdigest())
print("\n--- 130 ky tu dau ---"); print(PROMPT_PHIEU[:130])
print("\n--- 90 ky tu cuoi ---"); print(PROMPT_PHIEU[-90:])

## 6. Bộ chấm — **giữ y nguyên lượt A** để hai lượt so được với nhau

In [ ]:
import json, re

def chuan_hoa_ma_icd(raw_code):
    if not raw_code:
        return None
    code = str(raw_code).strip().upper().rstrip('.')
    m = re.match(r'^([A-Z]\d{2}(?:\d)?(?:\.\d{1,2})?)$', code)
    return m.group(1) if m else None


def extract_json_safe(raw):
    m = re.search(r'\{[^{}]*"top10_icd"\s*:\s*\[[^\]]*\][^{}]*\}', raw)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {'top10_icd': re.findall(r'\b([A-Z]\d{2}(?:\.\d{1,2})?)\b', raw)[:10]}


def cham_1_ca(llm_output_raw, gold3, gold_full, valid_3char):
    raw_list = extract_json_safe(llm_output_raw).get('top10_icd', [])
    top_10   = [c for c in (chuan_hoa_ma_icd(x) for x in raw_list) if c][:10]
    top_10_3 = [c[:3] for c in top_10]
    g3 = str(gold3).strip().upper()
    gf = str(gold_full).strip().upper()
    try:
        rank_3char = top_10_3.index(g3) + 1
    except ValueError:
        rank_3char = 0
    return {
        'top_10'              : ' || '.join(top_10),
        'top_10_3char'        : ' || '.join(top_10_3),
        'hit1'                : bool(top_10_3 and top_10_3[0] == g3),
        'hit5'                : g3 in top_10_3[:5],
        'hit10'               : g3 in top_10_3[:10],
        'hit1_full_icd'       : bool(top_10 and top_10[0] == gf),
        'rank_3char'          : rank_3char,
        'n_ma_parse_duoc'     : len(top_10),
        'n_valid_in_catalogue': sum(1 for c in top_10_3 if c in valid_3char),
    }

_r = cham_1_ca('{"top10_icd": ["K21.9","R10","XX","J32.","A09"]}', 'R10', 'R10.4', valid_codes_3char)
assert _r['rank_3char'] == 2 and _r['n_ma_parse_duoc'] == 4, _r
print("Bo cham OK (giong luot A):", _r)

## 7. Vá môi trường rồi nạp model — y nguyên lượt A

In [ ]:
# ===== VA TRUOC KHI NAP MODEL =====
import os, sys, shutil, tempfile

try:
    from transformers.utils import HF_MODULES_CACHE as _MOD_CACHE
except Exception:
    _MOD_CACHE = os.path.expanduser('~/.cache/huggingface/modules')
_tm = os.path.join(_MOD_CACHE, 'transformers_modules')
if os.path.isdir(_tm):
    shutil.rmtree(_tm, ignore_errors=True)
    print("Da xoa cache module dong:", _tm)
for _m in [m for m in list(sys.modules) if m.startswith('transformers_modules')]:
    del sys.modules[_m]

try:
    import triton_pre_mlir  # noqa: F401
    print("triton_pre_mlir: da co san")
except ImportError:
    goc = '/content' if os.path.isdir('/content') else tempfile.gettempdir()
    stub = os.path.join(goc, '_stub_triton')
    os.makedirs(os.path.join(stub, 'triton_pre_mlir'), exist_ok=True)
    with open(os.path.join(stub, 'triton_pre_mlir', '__init__.py'), 'w', encoding='utf-8') as f:
        f.write('# Stub rong: PhoGPT chay attn_impl="torch", nhanh triton khong bao gio import.\n')
    if stub not in sys.path:
        sys.path.insert(0, stub)
    import triton_pre_mlir  # noqa: F401
    print("triton_pre_mlir: da tao stub tai", stub)

import transformers.models.llama.modeling_llama as _ll
def _tao_lop_gia(ten):
    def __init__(self, *a, **k):
        raise RuntimeError(ten + " bi khoi tao that - PhoGPT le ra chay ALiBi (rope=False).")
    return type(ten, (object,), {'__init__': __init__})
_da_va = []
for _ten in ('LlamaRotaryEmbedding', 'LlamaLinearScalingRotaryEmbedding',
             'LlamaDynamicNTKScalingRotaryEmbedding'):
    if not hasattr(_ll, _ten):
        setattr(_ll, _ten, _tao_lop_gia(_ten)); _da_va.append(_ten)
print("Lop rotary da va:", ', '.join(_da_va) if _da_va else "khong thieu lop nao")

In [ ]:
DUNG_4BIT = False          # doi thanh True neu OOM

from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print(f"Dang tai: {MODEL_ID}  ({'4-bit NF4' if DUNG_4BIT else 'fp16'})")
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.init_device = 'cuda'
try:
    if isinstance(config.attn_config, dict):
        config.attn_config['attn_impl'] = 'torch'
    else:
        config.attn_config.attn_impl = 'torch'
except AttributeError:
    pass

kwargs = dict(config=config, trust_remote_code=True)
if DUNG_4BIT:
    kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
else:
    kwargs['torch_dtype'] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Nap model xong!")

## 8. Hàm sinh — **đúng mục 4.3 của phiếu**

Khác lượt A đúng một chỗ: **không** gọi `apply_chat_template`. Prompt đã tự mang cặp thẻ
`### Câu hỏi:` / `### Trả lời:` nên đưa thẳng vào tokenizer, đúng như phiếu viết.
`eos_token_id` cũng dùng `tokenizer.eos_token_id` như phiếu, không dùng danh sách
`TERMINATORS` của lượt A.

Cell in nguyên văn chuỗi vào model và **đếm cặp thẻ** — phải đúng 1, nếu 2 là thẻ đôi.

In [ ]:
def sinh_predictions(query_text, max_new_tokens=250):
    """Nguyen van muc 4.3 phieu: tokenizer tho, khong chat template."""
    prompt = PROMPT_PHIEU.format(query_text=query_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


_p = PROMPT_PHIEU.format(query_text=df_test.iloc[0]['query_text'])
print("So token dau vao:", len(tokenizer(_p).input_ids))
for _the in ['### Câu hỏi:', '### Trả lời:']:
    n = _p.count(_the)
    print(f"  {_the!r}: {n} lan" + ("  <-- LOI: the doi!" if n > 1 else ""))
assert _p.count('### Câu hỏi:') == 1 and _p.count('### Trả lời:') == 1

_thu = sinh_predictions(df_test.iloc[0]['query_text'])
print("\nOutput thu nghiem:\n", _thu[:400])
assert _thu, "Model tra ve rong."

## 9. Dry run 10 ca — cùng bốn cổng của lượt A

Cổng "chép ví dụ" là cổng quan trọng nhất ở đây: nếu lượt B **cũng** chép thì việc thay
prompt không phải nguyên nhân, và kết luận của lượt A đứng vững.

In [ ]:
import time

MA_VI_DU_3 = ['K35', 'K52', 'K37', 'K59', 'R10', 'K85', 'R11', 'A09', 'K80', 'R50']

def ty_le_trung(top_10_3char, bo_ma):
    ds = [c.strip() for c in str(top_10_3char).split('||') if c.strip()]
    if len(ds) < 3:
        return 0.0
    return sum(1 for c in ds if c in set(bo_ma)) / len(ds)

def chep_vi_du(s):
    return ty_le_trung(s, MA_VI_DU_3) >= 0.8

print("=" * 70); print("DRY RUN 10 CA - PROMPT CUA PHIEU"); print("=" * 70)
t0, tong_valid, tong_ma, n_parse_ok, n_hit1, n_chep = time.time(), 0, 0, 0, 0, 0
for _, row in df_test.head(10).iterrows():
    raw = sinh_predictions(row['query_text'])
    sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
    tong_valid += sc['n_valid_in_catalogue']
    tong_ma    += max(sc['n_ma_parse_duoc'], 1)
    n_parse_ok += 1 if sc['n_ma_parse_duoc'] > 0 else 0
    n_hit1     += 1 if sc['hit1'] else 0
    la_chep     = chep_vi_du(sc['top_10_3char'])
    n_chep     += 1 if la_chep else 0
    print(f"\n[{row['case_id']}] gold3={row['gold3']}")
    print(f"  raw   : {raw[:150]}")
    print(f"  top10 : {sc['top_10_3char']}")
    print(f"  Hit@1={sc['hit1']} valid={sc['n_valid_in_catalogue']}/{sc['n_ma_parse_duoc']}"
          f"{'   *** CHEP LAI VI DU ***' if la_chep else ''}")

giay = (time.time() - t0) / 10
print("\n" + "=" * 70)
print(f"JSON parse ra >=1 ma  : {n_parse_ok}/10   (cong: >= 8)")
print(f"Ty le ma hop le       : {100*tong_valid/tong_ma:.1f}%   (cong: >= 70)")
print(f"CHEP VI DU            : {n_chep}/10   (luot A: 10/10)")
print(f"Hit@1 tren 10 ca      : {n_hit1}/10   (luot A: 1/10, do trung ngau nhien)")
print(f"Toc do                : {giay:.1f} giay/ca -> uoc {giay*181/60:.0f} phut")
print("\n=> Luot B CUNG chep vi du. Ket luan luot A vung, chay full de co so chinh thuc."
      if n_chep >= 8 else
      "\n=> Luot B KHAC luot A. Doc ky output roi bao thay TRUOC khi chay full.")

## 10. Chạy full 181 ca

In [ ]:
from tqdm.auto import tqdm

t0, results = time.time(), []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="181 ca - prompt phieu"):
    raw = sinh_predictions(row['query_text'])
    sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
    results.append({
        'case_id': row['case_id'], 'source_channel': row['source_channel'],
        'gold3': str(row['gold3']).strip().upper(),
        'gold_full': str(row['gold']).strip().upper(),
        'voi_toi_duoc': row['voi_toi_duoc'], 'llm_raw': raw, **sc})

df_out = pd.DataFrame(results)
PHUT_CHAY = (time.time() - t0) / 60
df_out.to_csv(f'{OUT}/tn4b_perquery_phogpt_promptphieu.csv', index=False, encoding='utf-8-sig')
print(f"\nXong {len(df_out)} ca trong {PHUT_CHAY:.1f} phut")
print(f"VRAM dinh: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print(f"So ca parse duoc 0 ma: {(df_out['n_ma_parse_duoc'] == 0).sum()}")

## 11. Chỉ số tổng hợp

In [ ]:
from scipy import stats

def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return (0.0, 0.0)
    z = stats.norm.ppf(1 - alpha/2); p = k/n
    den = 1 + z**2/n
    c = (p + z**2/(2*n))/den
    h = z*((p*(1-p)/n + z**2/(4*n**2))**0.5)/den
    return (max(0, (c-h)*100), min(100, (c+h)*100))

n = len(df_out)
reach = df_out[df_out['voi_toi_duoc'] == True]
outr  = df_out[df_out['voi_toi_duoc'] == False]
hit1_n = int(df_out['hit1'].sum())
lo, hi = wilson_ci(hit1_n, n)
lo_o, hi_o = wilson_ci(int(outr['hit1'].sum()), len(outr))
mrr = df_out['rank_3char'].apply(lambda x: 1/x if x > 0 else 0).mean()

summary = {
    'system': 'PhoGPT-4B-Chat_unconstrained_promptPHIEU', 'model_id': MODEL_ID, 'n': n,
    'hit1_pct': round(100*hit1_n/n, 2), 'hit1_ci': f'[{lo:.2f}, {hi:.2f}]',
    'hit5_pct': round(100*df_out['hit5'].sum()/n, 2),
    'hit10_pct': round(100*df_out['hit10'].sum()/n, 2), 'mrr': round(float(mrr), 4),
    'n_reachable': len(reach),
    'hit1_reachable_pct': round(100*reach['hit1'].sum()/len(reach), 2),
    'n_outreach': len(outr),
    'hit1_outreach_pct': round(100*outr['hit1'].sum()/len(outr), 2),
    'hit1_outreach_ci': f'[{lo_o:.2f}, {hi_o:.2f}]',
    'hit1_full_icd_pct': round(100*df_out['hit1_full_icd'].sum()/n, 2),
    'avg_valid_top10': round(float(df_out['n_valid_in_catalogue'].mean()), 2),
    'pct_ma_hop_le': round(100*df_out['n_valid_in_catalogue'].sum()
                           / max(df_out['n_ma_parse_duoc'].sum(), 1), 2),
    'so_luong_bit': '4bit_nf4' if DUNG_4BIT else 'fp16',
    'vram_dinh_GB': round(torch.cuda.max_memory_allocated()/1e9, 2),
    'phut_chay': round(PHUT_CHAY, 1),
    'prompt_sha256': hashlib.sha256(PROMPT_PHIEU.encode()).hexdigest(),
}
pd.DataFrame([summary]).to_csv(f'{OUT}/tn4b_summary.csv', index=False, encoding='utf-8-sig')
for k, v in summary.items():
    print(f"{k:<24}: {v}")

## 12. So sánh lượt A ↔ lượt B — phần trả lời câu hỏi của cả notebook này

In [ ]:
so_ma_B = len({c.strip() for s in df_out['top_10_3char'] for c in str(s).split('||') if c.strip()})
chep_B  = int(df_out['top_10_3char'].apply(chep_vi_du).sum())
hit1_B  = summary['hit1_pct']

print("=" * 76)
print(f"{'Chi so':<36}{'Luot A (prompt TN3)':>19}{'Luot B (prompt phieu)':>21}")
print("=" * 76)

if CO_LUOT_A:
    dfA = pd.read_csv(f'{DATA}/tn4a_perquery_phogpt.csv')
    so_ma_A = len({c.strip() for s in dfA['top_10_3char'] for c in str(s).split('||') if c.strip()})
    chep_A  = int(dfA['top_10_3char'].apply(chep_vi_du).sum())
    hit1_A  = round(100*int(dfA['hit1'].astype(bool).sum())/181, 2)
    print(f"{'Hit@1 (%)':<36}{hit1_A:>19}{hit1_B:>21}")
    print(f"{'So ma 3 ky tu khac nhau / 181 ca':<36}{so_ma_A:>19}{so_ma_B:>21}")
    print(f"{'So ca chep vi du':<36}{str(chep_A) + '/181':>19}{str(chep_B) + '/181':>21}")
    a = dfA.set_index('case_id')['top_10_3char'].astype(str)
    b = df_out.set_index('case_id')['top_10_3char'].astype(str)
    chung = a.index.intersection(b.index)
    giong = sum(1 for i in chung if a[i] == b[i])
    print(f"\nSo ca hai luot cho top-10 GIONG HET: {giong}/{len(chung)}")
else:
    print(f"{'Hit@1 (%)':<36}{'(thieu tep luot A)':>19}{hit1_B:>21}")
    print(f"{'So ma 3 ky tu khac nhau':<36}{'-':>19}{so_ma_B:>21}")
    print(f"{'So ca chep vi du':<36}{'-':>19}{str(chep_B) + '/181':>21}")

print("\n" + "=" * 76)
if chep_B >= 145:
    print("KET LUAN: prompt cua phieu CUNG bi sao chep vi du.")
    print("  => Hien tuong khong do nhom thay prompt. Ket luan luot A vung.")
    print("  => Giu nguyen so 0,55% da dua vao Bang 12.")
elif hit1_B >= 5:
    print("KET LUAN: prompt cua phieu cho ket qua KHAC HAN.")
    print("  => Luot A sai vi da thay prompt. PHAI thay so Bang 12 bang so luot B.")
    print("  => Bao thay ngay truoc khi sua bai.")
else:
    print("KET LUAN: khong roi vao hai kich ban ro rang. Doc ky output roi bao thay.")
print("=" * 76)

## 13. Băm SHA-256 và tải về

In [ ]:
import hashlib, shutil

dong = []
for f in sorted(os.listdir(OUT)):
    if f.endswith('SHA256.txt'):
        continue
    with open(f'{OUT}/{f}', 'rb') as fh:
        dong.append(f'{hashlib.sha256(fh.read()).hexdigest()}  {f}')
with open(f'{OUT}/tn4b_SHA256.txt', 'w', encoding='utf-8', newline='\n') as fh:
    fh.write('\n'.join(dong) + '\n')
print('\n'.join(dong))

shutil.make_archive('/content/TN4B_KetQua', 'zip', OUT)
from google.colab import files
files.download('/content/TN4B_KetQua.zip')